In [ ]:
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Data management
from dataclasses import dataclass, field
import h5py as hf
import glob
import pandas as pd

In [ ]:
@dataclass
class Architecture:
    # Architecture Description
    architecture: str
    width: int
    depth: int
    activation: str

    # To store references
    files: list = field(default_factory=list)

@dataclass
class Measurement:
    # References
    file: str
    dataset: str

    # Measurements
    loss: np.ndarray
    entropy: np.ndarray
    trace: np.ndarray


class ArchictureStudy:
    """ 
    A single architecture study.
    """

    def __init__(self, architecture: Architecture):
        """ 
        Construct a new experiment.

        Parameters
        ----------
        architecture : Architecture
            The kind of architecture this experiment belongs to.
        """
        self.architecture = architecture

        self._is_populated = False  # There is no data in the clas yet.

        # Stuff you shouldn't touch
        self._train_loss = []
        self._train_entropy = []
        self._train_trace = []

        self._test_loss = []
        self._test_entropy = []
        self._test_trace = []

    def _read_in_data(self) -> list:
        """ 
        Read from all HDF5 files.
        """
        measurements = []
        for file in self.architecture.files:
            with hf.File(file, "r") as db:
                loss = db["loss"][:]
                entropy = db["entropy"][:]
                trace = db["trace"][:]
            
            if file.split("/")[-1].split("_")[0] == "train":
                dataset = "train"
            else:
                dataset = "test"

            measurements.append(
                Measurement(
                    file=file,
                    dataset=dataset,
                    loss=loss,
                    entropy=entropy,
                    trace=trace,
                )
            )
        
        return measurements

    def _consolidate_files(self):
        """ 
        Populate the properties
        """
        if not self._is_populated:
            measurements = self._read_in_data()
            self._train_loss = [
                item.loss for item in measurements if item.dataset == "train"
            ]
            self._train_entropy = [
                item.entropy for item in measurements if item.dataset == "train"
            ]
            self._train_trace = [
                item.trace for item in measurements if item.dataset == "train"
            ]
            self._test_loss = [
                item.loss for item in measurements if item.dataset == "test"
            ]
            self._test_entropy = [
                item.entropy for item in measurements if item.dataset == "test"
            ]
            self._test_trace = [
                item.trace for item in measurements if item.dataset == "test"
            ]

            self._is_populated = True
        else:
            pass

    @property   
    def train_loss(self) -> tuple:
        self._consolidate_files()
        return np.mean(self._train_loss, axis=0), np.std(self._train_loss, axis=0)

    @property   
    def train_entropy(self) -> tuple:
        self._consolidate_files()
        return np.mean(self._train_entropy, axis=0), np.std(self._train_entropy, axis=0)

    @property   
    def train_trace(self) -> tuple:
        self._consolidate_files()
        return np.mean(self._train_trace, axis=0), np.std(self._train_trace, axis=0)

    @property   
    def test_loss(self) -> tuple:
        self._consolidate_files()
        return np.mean(self._test_loss, axis=0), np.std(self._test_loss, axis=0)

    @property   
    def test_entropy(self) -> tuple:
        self._consolidate_files()
        return np.mean(self._test_entropy, axis=0), np.std(self._test_entropy, axis=0)

    @property   
    def test_trace(self) -> tuple:
        self._consolidate_files()
        return np.mean(self._test_trace, axis=0), np.std(self._test_trace, axis=0)


In [ ]:
# Populate the architecture classes and then the ArchitectureStudy list
studies = []
architectures = {}

widths=[4, 12, 50, 100, 500, 1000]
depths=[1, 2, 3, 5, 10]
activations=["relu", "tanh"]

all_files = glob.glob("adam-slow/*.h5")

# Construct the data
for depth in depths:
    architectures[depth] = {}
    for width in widths:
        architectures[depth][width] = {}
        for activation in activations:
            architectures[depth][width][activation] = Architecture(
                    architecture="dense",
                    width=width,
                    depth=depth,
                    activation=activation
                )


for name in all_files:
    model_params = name.split("/")[-1].split('_')
    width = int(model_params[2])
    depth = int(model_params[3])
    activation = model_params[4]
    architectures[depth][width][activation].files.append(name)


for item in list(list(pd.json_normalize(architectures).T.to_dict().values())[0].values()):
    studies.append(
        ArchictureStudy(
            architecture=item
        )
    )
 

In [ ]:
epochs = np.linspace(1, 500, 500, dtype=int)

## Evolution Plots

In [ ]:
# Global Configuration
fig, axes = plt.subplots(3, 3, figsize=(8.0, 8.0))
alpha = 0.2

line_styles = ['-', '--']
train_colours = ['#101419', '#30C5FF']
test_colours = ['#09BC8A', '#F7E733']

# Activation Scaling
row = 0
ax = axes[row]
activations = ["relu", "tanh"]
d_width = 50
d_depth = 2
for activation in activations:
    for study in studies:
        if study.architecture.activation == activation and study.architecture.width == d_width and study.architecture.depth == d_depth:
            ax[0].plot(
                epochs, study.train_loss[0], color=train_colours[activations.index(activation)]
            )
            ax[0].fill_between(
                epochs, study.train_loss[0] - study.train_loss[1], study.train_loss[0] + study.train_loss[1], alpha=alpha, color=train_colours[activations.index(activation)]
            )
            ax[0].plot(
                epochs, study.test_loss[0], color=test_colours[activations.index(activation)]
            )
            ax[0].fill_between(
                epochs, study.test_loss[0] - study.test_loss[1], study.test_loss[0] + study.test_loss[1], alpha=alpha, color=test_colours[activations.index(activation)]
            )
            ax[1].plot(
                epochs, study.train_entropy[0], color=train_colours[activations.index(activation)]
            )
            ax[1].fill_between(
                epochs, study.train_entropy[0] - study.train_entropy[1], study.train_entropy[0] + study.train_entropy[1], alpha=alpha, color=train_colours[activations.index(activation)]
            )
            ax[1].plot(
                epochs, study.test_entropy[0], color=test_colours[activations.index(activation)]
            )
            ax[1].fill_between(
                epochs, study.test_entropy[0] - study.test_entropy[1], study.test_entropy[0] + study.test_entropy[1], alpha=alpha, color=test_colours[activations.index(activation)]
            )
            ax[2].plot(
                epochs, study.train_trace[0], color=train_colours[activations.index(activation)]
            )
            ax[2].fill_between(
                epochs, study.train_trace[0] - study.train_trace[1], study.train_trace[0] + study.train_trace[1], alpha=alpha, color=train_colours[activations.index(activation)]
            )
            ax[2].plot(
                epochs, study.test_trace[0], color=test_colours[activations.index(activation)]
            )
            ax[2].fill_between(
                epochs, study.test_trace[0] - study.test_trace[1], study.test_trace[0] + study.test_trace[1], alpha=alpha, color=test_colours[activations.index(activation)]
            )

blue_patch = mpatches.Patch(color='#101419', label=f'Train ReLU')
red_patch = mpatches.Patch(color='#30C5FF', label='Train Tanh')
blue_patch2 = mpatches.Patch(color='#09BC8A', label='Test ReLU')
red_patch2 = mpatches.Patch(color='#F7E733', label='Test Tanh')
ax[2].legend(handles=[blue_patch, red_patch, blue_patch2, red_patch2], title="Activation")

ax[1].set_title("Activation Scaling (32, 2, x)")
ax[0].set_yscale("log")
ax[2].set_yscale("log")
# ax[0].set_xlabel("Epochs")
# ax[1].set_xlabel("Epochs")
# ax[2].set_xlabel("Epochs")
ax[0].set_ylabel("Loss")
ax[1].set_ylabel("Entropy")
ax[2].set_ylabel("Trace")
ax[0].set_xscale("log")
ax[1].set_xscale("log")
ax[2].set_xscale("log")
# Width Scaling
row = 1
ax = axes[row]
widths = [12, 1000]
d_depth = 3
d_activation = "relu"
for width in widths:
    for study in studies:
        if study.architecture.width == width and study.architecture.depth == d_depth and study.architecture.activation == d_activation:
            ax[0].plot(
                epochs, study.train_loss[0], color=train_colours[widths.index(width)]
            )
            ax[0].fill_between(
                epochs, study.train_loss[0] - study.train_loss[1], study.train_loss[0] + study.train_loss[1], alpha=alpha, color=train_colours[widths.index(width)]
            )
            ax[0].plot(
                epochs, study.test_loss[0], color=test_colours[widths.index(width)]
            )
            ax[0].fill_between(
                epochs, study.test_loss[0] - study.test_loss[1], study.test_loss[0] + study.test_loss[1], alpha=alpha, color=test_colours[widths.index(width)]
            )
            ax[1].plot(
                epochs, study.train_entropy[0], color=train_colours[widths.index(width)]
            )
            ax[1].fill_between(
                epochs, study.train_entropy[0] - study.train_entropy[1], study.train_entropy[0] + study.train_entropy[1], alpha=alpha, color=train_colours[widths.index(width)]
            )
            ax[1].plot(
                epochs, study.test_entropy[0], color=test_colours[widths.index(width)]
            )
            ax[1].fill_between(
                epochs, study.test_entropy[0] - study.test_entropy[1], study.test_entropy[0] + study.test_entropy[1], alpha=alpha, color=test_colours[widths.index(width)]
            )
            ax[2].plot(
                epochs, study.train_trace[0], color=train_colours[widths.index(width)]
            )
            ax[2].fill_between(
                epochs, study.train_trace[0] - study.train_trace[1], study.train_trace[0] + study.train_trace[1], alpha=alpha, color=train_colours[widths.index(width)]
            )
            ax[2].plot(
                epochs, study.test_trace[0], color=test_colours[widths.index(width)]
            )
            ax[2].fill_between(
                epochs, study.test_trace[0] - study.test_trace[1], study.test_trace[0] + study.test_trace[1], alpha=alpha, color=test_colours[widths.index(width)]
            )
   
blue_patch = mpatches.Patch(color='#101419', label=f'Train {widths[0]}')
red_patch = mpatches.Patch(color='#30C5FF', label=f'Train {widths[1]}')
blue_patch2 = mpatches.Patch(color='#09BC8A', label=f'Test {widths[0]}')
red_patch2 = mpatches.Patch(color='#F7E733', label=f'Test {widths[1]}')
ax[2].legend(handles=[blue_patch, red_patch, blue_patch2, red_patch2], title="Width")

ax[1].set_title(f"Width Scaling (x, {d_depth}, ReLU)")
ax[0].set_yscale("log")
ax[2].set_yscale("log")
# ax[0].set_xlabel("Epochs")
# ax[1].set_xlabel("Epochs")
# ax[2].set_xlabel("Epochs")
ax[0].set_ylabel("Loss")
ax[1].set_ylabel("Entropy")
ax[2].set_ylabel("Trace")
ax[0].set_xscale("log")
ax[1].set_xscale("log")
ax[2].set_xscale("log")
# Depth Scaling
row = 2
ax = axes[row]
depths = [2, 10]
d_width = 100
d_activation = "relu"
for depth in depths:
    for study in studies:
        if study.architecture.depth == depth and study.architecture.width == d_width and study.architecture.activation == d_activation:
            ax[0].plot(
                epochs, study.train_loss[0], color=train_colours[depths.index(depth)]
            )
            ax[0].fill_between(
                epochs, study.train_loss[0] - study.train_loss[1], study.train_loss[0] + study.train_loss[1], alpha=alpha, color=train_colours[depths.index(depth)]
            )
            ax[0].plot(
                epochs, study.test_loss[0], color=test_colours[depths.index(depth)]
            )
            ax[0].fill_between(
                epochs, study.test_loss[0] - study.test_loss[1], study.test_loss[0] + study.test_loss[1], alpha=alpha, color=test_colours[depths.index(depth)]
            )
            ax[1].plot(
                epochs, study.train_entropy[0], color=train_colours[depths.index(depth)]
            )
            ax[1].fill_between(
                epochs, study.train_entropy[0] - study.train_entropy[1], study.train_entropy[0] + study.train_entropy[1], alpha=alpha, color=train_colours[depths.index(depth)]
            )
            ax[1].plot(
                epochs, study.test_entropy[0], color=test_colours[depths.index(depth)]
            )
            ax[1].fill_between(
                epochs, study.test_entropy[0] - study.test_entropy[1], study.test_entropy[0] + study.test_entropy[1], alpha=alpha, color=test_colours[depths.index(depth)]
            )
            ax[2].plot(
                epochs, study.train_trace[0], color=train_colours[depths.index(depth)]
            )
            ax[2].fill_between(
                epochs, study.train_trace[0] - study.train_trace[1], study.train_trace[0] + study.train_trace[1], alpha=alpha, color=train_colours[depths.index(depth)]
            )
            ax[2].plot(
                epochs, study.test_trace[0], color=test_colours[depths.index(depth)]
            )
            ax[2].fill_between(
                epochs, study.test_trace[0] - study.test_trace[1], study.test_trace[0] + study.test_trace[1], alpha=alpha, color=test_colours[depths.index(depth)]
            )

blue_patch = mpatches.Patch(color='#101419', label=f'Train {depths[0]}')
red_patch = mpatches.Patch(color='#30C5FF', label=f'Train {depths[1]}')
blue_patch2 = mpatches.Patch(color='#09BC8A', label=f'Test {depths[0]}')
red_patch2 = mpatches.Patch(color='#F7E733', label=f'Test {depths[1]}')
ax[2].legend(handles=[blue_patch, red_patch, blue_patch2, red_patch2], title="Depth")

ax[1].set_title(f"Depth Scaling ({d_width}, x, ReLU)")
ax[0].set_yscale("log")
ax[2].set_yscale("log")
ax[0].set_xlabel("Epochs")
ax[1].set_xlabel("Epochs")
ax[2].set_xlabel("Epochs")
ax[0].set_ylabel("Loss")
ax[1].set_ylabel("Entropy")
ax[2].set_ylabel("Trace")
ax[0].set_xscale("log")
ax[1].set_xscale("log")
ax[2].set_xscale("log")

plt.tight_layout()
plt.savefig("dense-mnist-combined.pdf")
plt.show()

In [ ]:
# Global Configuration
fig, axes = plt.subplots(3, 3, figsize=(8.3, 11.7))
alpha = 0.2

line_styles = ['-', '--']
train_colours = ['#101419', '#30C5FF']
test_colours = ['#09BC8A', '#F7E733']

# Activation Scaling
row = 0
ax = axes[row]
activations = ["relu", "tanh"]
d_width = 64
d_depth = 3
for activation in activations:
    for study in studies:
        if study.architecture.activation == activation and study.architecture.width == d_width and study.architecture.depth == d_depth:
            ax[0].plot(
                epochs, study.train_loss[0], color=train_colours[activations.index(activation)]
            )
            ax[0].fill_between(
                epochs, study.train_loss[0] - study.train_loss[1], study.train_loss[0] + study.train_loss[1], alpha=alpha, color=train_colours[activations.index(activation)]
            )
            ax[0].plot(
                epochs, study.test_loss[0], color=test_colours[activations.index(activation)]
            )
            ax[0].fill_between(
                epochs, study.test_loss[0] - study.test_loss[1], study.test_loss[0] + study.test_loss[1], alpha=alpha, color=test_colours[activations.index(activation)]
            )
            ax[1].plot(
                study.train_loss[0], study.train_entropy[0], color=train_colours[activations.index(activation)]
            )
            ax[1].fill_between(
                study.train_loss[0], study.train_entropy[0] - study.train_entropy[1], study.train_entropy[0] + study.train_entropy[1], alpha=alpha, color=train_colours[activations.index(activation)]
            )
            ax[1].plot(
                study.test_loss[0], study.test_entropy[0], color=test_colours[activations.index(activation)]
            )
            ax[1].fill_between(
                study.test_loss[0], study.test_entropy[0] - study.test_entropy[1], study.test_entropy[0] + study.test_entropy[1], alpha=alpha, color=test_colours[activations.index(activation)]
            )
            ax[2].plot(
                study.train_loss[0], study.train_trace[0], color=train_colours[activations.index(activation)]
            )
            ax[2].fill_between(
                study.train_loss[0], study.train_trace[0] - study.train_trace[1], study.train_trace[0] + study.train_trace[1], alpha=alpha, color=train_colours[activations.index(activation)]
            )
            ax[2].plot(
                study.test_loss[0], study.test_trace[0], color=test_colours[activations.index(activation)]
            )
            ax[2].fill_between(
                study.test_loss[0], study.test_trace[0] - study.test_trace[1], study.test_trace[0] + study.test_trace[1], alpha=alpha, color=test_colours[activations.index(activation)]
            )

blue_patch = mpatches.Patch(color='#101419', label=f'Train ReLU')
red_patch = mpatches.Patch(color='#30C5FF', label='Train Tanh')
blue_patch2 = mpatches.Patch(color='#09BC8A', label='Test ReLU')
red_patch2 = mpatches.Patch(color='#F7E733', label='Test Tanh')
ax[1].legend(handles=[blue_patch, red_patch, blue_patch2, red_patch2], title="Activation")

ax[1].set_title("Activation Scaling (32, 2, x)")
ax[0].set_yscale("log")
ax[2].set_yscale("log")
ax[1].set_xscale("log")
ax[2].set_xscale("log")
ax[1].invert_xaxis()
ax[2].invert_xaxis()
ax[0].set_ylabel("Loss")
ax[1].set_ylabel("Entropy")
ax[2].set_ylabel("Trace")
            
# Width Scaling
row = 1
ax = axes[row]
widths = [12, 64]
d_depth = 3
d_activation = "tanh"
for width in widths:
    for study in studies:
        if study.architecture.width == width and study.architecture.depth == d_depth and study.architecture.activation == d_activation:
            ax[0].plot(
                epochs, study.train_loss[0], color=train_colours[widths.index(width)]
            )
            ax[0].fill_between(
                epochs, study.train_loss[0] - study.train_loss[1], study.train_loss[0] + study.train_loss[1], alpha=alpha, color=train_colours[widths.index(width)]
            )
            ax[0].plot(
                epochs, study.test_loss[0], color=test_colours[widths.index(width)]
            )
            ax[0].fill_between(
                epochs, study.test_loss[0] - study.test_loss[1], study.test_loss[0] + study.test_loss[1], alpha=alpha, color=test_colours[widths.index(width)]
            )
            ax[1].plot(
                study.train_loss[0], study.train_entropy[0], color=train_colours[widths.index(width)]
            )
            ax[1].fill_between(
                study.train_loss[0], study.train_entropy[0] - study.train_entropy[1], study.train_entropy[0] + study.train_entropy[1], alpha=alpha, color=train_colours[widths.index(width)]
            )
            ax[1].plot(
                study.test_loss[0], study.test_entropy[0], color=test_colours[widths.index(width)]
            )
            ax[1].fill_between(
                study.test_loss[0], study.test_entropy[0] - study.test_entropy[1], study.test_entropy[0] + study.test_entropy[1], alpha=alpha, color=test_colours[widths.index(width)]
            )
            ax[2].plot(
                study.train_loss[0], study.train_trace[0], color=train_colours[widths.index(width)]
            )
            ax[2].fill_between(
                study.train_loss[0], study.train_trace[0] - study.train_trace[1], study.train_trace[0] + study.train_trace[1], alpha=alpha, color=train_colours[widths.index(width)]
            )
            ax[2].plot(
                study.test_loss[0], study.test_trace[0], color=test_colours[widths.index(width)]
            )
            ax[2].fill_between(
                study.test_loss[0], study.test_trace[0] - study.test_trace[1], study.test_trace[0] + study.test_trace[1], alpha=alpha, color=test_colours[widths.index(width)]
            )
   
blue_patch = mpatches.Patch(color='#101419', label=f'Train {widths[0]}')
red_patch = mpatches.Patch(color='#30C5FF', label=f'Train {widths[1]}')
blue_patch2 = mpatches.Patch(color='#09BC8A', label=f'Test {widths[0]}')
red_patch2 = mpatches.Patch(color='#F7E733', label=f'Test {widths[1]}')
ax[1].legend(handles=[blue_patch, red_patch, blue_patch2, red_patch2], title="Width")

ax[1].set_title(f"Width Scaling (x, {d_depth}, ReLU)")
ax[0].set_yscale("log")
ax[2].set_yscale("log")
ax[1].set_xscale("log")
ax[2].set_xscale("log")
ax[1].invert_xaxis()
ax[2].invert_xaxis()
ax[0].set_ylabel("Loss")
ax[1].set_ylabel("Entropy")
ax[2].set_ylabel("Trace")

# Depth Scaling
row = 2
ax = axes[row]
depths = [1, 3]
d_width = 64
d_activation = "tanh"
for depth in depths:
    for study in studies:
        if study.architecture.depth == depth and study.architecture.width == d_width and study.architecture.activation == d_activation:
            ax[0].plot(
                epochs, study.train_loss[0], color=train_colours[depths.index(depth)]
            )
            ax[0].fill_between(
                epochs, study.train_loss[0] - study.train_loss[1], study.train_loss[0] + study.train_loss[1], alpha=alpha, color=train_colours[depths.index(depth)]
            )
            ax[0].plot(
                epochs, study.test_loss[0], color=test_colours[depths.index(depth)]
            )
            ax[0].fill_between(
                epochs, study.test_loss[0] - study.test_loss[1], study.test_loss[0] + study.test_loss[1], alpha=alpha, color=test_colours[depths.index(depth)]
            )
            ax[1].plot(
                study.train_loss[0], study.train_entropy[0], color=train_colours[depths.index(depth)]
            )
            ax[1].fill_between(
                study.train_loss[0], study.train_entropy[0] - study.train_entropy[1], study.train_entropy[0] + study.train_entropy[1], alpha=alpha, color=train_colours[depths.index(depth)]
            )
            ax[1].plot(
                study.test_loss[0], study.test_entropy[0], color=test_colours[depths.index(depth)]
            )
            ax[1].fill_between(
                study.test_loss[0], study.test_entropy[0] - study.test_entropy[1], study.test_entropy[0] + study.test_entropy[1], alpha=alpha, color=test_colours[depths.index(depth)]
            )
            ax[2].plot(
                study.train_loss[0], study.train_trace[0], color=train_colours[depths.index(depth)]
            )
            ax[2].fill_between(
                study.train_loss[0], study.train_trace[0] - study.train_trace[1], study.train_trace[0] + study.train_trace[1], alpha=alpha, color=train_colours[depths.index(depth)]
            )
            ax[2].fill_between(
                study.test_loss[0], study.test_trace[0], color=test_colours[depths.index(depth)]
            )
            ax[2].fill_between(
                study.test_loss[0], study.test_trace[0] - study.test_trace[1], study.test_trace[0] + study.test_trace[1], alpha=alpha, color=test_colours[depths.index(depth)]
            )

blue_patch = mpatches.Patch(color='#101419', label=f'Train {depths[0]}')
red_patch = mpatches.Patch(color='#30C5FF', label=f'Train {depths[1]}')
blue_patch2 = mpatches.Patch(color='#09BC8A', label=f'Test {depths[0]}')
red_patch2 = mpatches.Patch(color='#F7E733', label=f'Test {depths[1]}')
ax[1].legend(handles=[blue_patch, red_patch, blue_patch2, red_patch2], title="Depth")

ax[1].set_title(f"Depth Scaling ({d_width}, x, ReLU)")
ax[0].set_yscale("log")
ax[2].set_yscale("log")
ax[1].set_xscale("log")
ax[2].set_xscale("log")
ax[0].set_xlabel("Epochs")
ax[1].set_xlabel("Epochs")
ax[2].set_xlabel("Epochs")
ax[0].set_ylabel("Loss")
ax[1].set_ylabel("Entropy")
ax[2].set_ylabel("Trace")
ax[1].invert_xaxis()
ax[2].invert_xaxis()


plt.tight_layout()
plt.show()